# Text Chunking, Embeddings, and ChromaDB Vector Store

Pipeline: load saved transcripts → chunk with overlap → embed via OpenAI → store in ChromaDB → test retriever

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("../.env"))

TRANSCRIPTS_DIR = Path("../data/transcripts")
VECTORSTORE_DIR = Path("../data/vectorstore")
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

print("Env loaded.")
print(f"Transcripts path: {TRANSCRIPTS_DIR.resolve()}")
print(f"Vectorstore path: {VECTORSTORE_DIR.resolve()}")

Env loaded.
Transcripts path: /home/lopoa/projects/Project-Youtube-QA-Bot/data/transcripts
Vectorstore path: /home/lopoa/projects/Project-Youtube-QA-Bot/data/vectorstore


## 1. Load Transcripts

Walk every category folder and pull each `.json` file. We use the JSON (not the `.txt`) because it already has the metadata we need — title, category, video ID.

In [2]:
transcripts = []

for json_file in sorted(TRANSCRIPTS_DIR.rglob("*.json")):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    raw = data.get("transcript", data.get("text", ""))

    # transcript field may be a list of segment dicts — join them into plain text
    if isinstance(raw, list):
        text = " ".join(seg.get("text", "") for seg in raw)
    else:
        text = raw

    transcripts.append({
        "text":     text,
        "title":    data.get("title", json_file.stem),
        "category": data.get("category", json_file.parent.name),
        "video_id": data.get("video_id", ""),
        "source":   str(json_file),
    })

print(f"Loaded {len(transcripts)} transcripts")
for t in transcripts:
    print(f"  [{t['category']}] {t['title']} — {len(t['text'])} chars")

Loaded 9 transcripts
  [education] gradient_descent_explained — 15797 chars
  [education] neural_networks_explained — 18430 chars
  [education] transformers_explained — 12746 chars
  [entertainment] bad_bunny_hot_ones — 11441 chars
  [entertainment] erwin_speech_aot — 1199 chars
  [entertainment] fantano_igor_review — 15774 chars
  [tech_ai] langchain_crash_course — 36394 chars
  [tech_ai] openai_gpt4_announcement — 56915 chars
  [tech_ai] rag_explained — 50914 chars


## 2. Chunk the Transcripts

Spoken text is looser and more repetitive than written text, so we use a moderate chunk size with enough overlap to preserve context across boundaries. `RecursiveCharacterTextSplitter` splits on natural breaks first (paragraphs, sentences) before falling back to character count.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_chunks = []

for t in transcripts:
    raw_chunks = splitter.split_text(t["text"])
    for i, chunk in enumerate(raw_chunks):
        all_chunks.append(
            Document(
                page_content=chunk,
                metadata={
                    "title":    t["title"],
                    "category": t["category"],
                    "video_id": t["video_id"],
                    "source":   t["source"],
                    "chunk_id": i,
                },
            )
        )

print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk length: {sum(len(c.page_content) for c in all_chunks) // len(all_chunks)} chars")
print("\nSample chunk:")
print(all_chunks[0].page_content)
print("Metadata:", all_chunks[0].metadata)

/home/lopoa/projects/Project-Youtube-QA-Bot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 514
Avg chunk length: 458 chars

Sample chunk:
Hey everyone, Grant here. This is the first video in a series on the essence of calculus, and I'll be publishing the following videos once per day for the next 10 days. The goal here, as the name suggests, is to really get the heart of the subject out in one binge-watchable set. But with a topic that's as broad as calculus, there's a lot of things that can mean, so here's what I have in mind specifically
Metadata: {'title': 'gradient_descent_explained', 'category': 'education', 'video_id': 'WUvTyaaNkzM', 'source': '../data/transcripts/education/gradient_descent_explained.json', 'chunk_id': 0}


## 3. Chunk Distribution Check

Quick sanity check — see how chunks are spread across categories before we commit to embeddings (each chunk = one API call).

In [4]:
from collections import Counter

category_counts = Counter(c.metadata["category"] for c in all_chunks)

print("Chunks per category:")
for cat, count in sorted(category_counts.items()):
    print(f"  {cat}: {count}")

print(f"\nTotal chunks to embed: {len(all_chunks)}")
est_cost = len(all_chunks) * 500 / 1_000_000 * 0.02
print(f"Estimated embedding cost: ~${est_cost:.4f}")

Chunks per category:
  education: 122
  entertainment: 66
  tech_ai: 326

Total chunks to embed: 514
Estimated embedding cost: ~$0.0051


## 4. Generate Embeddings and Build the ChromaDB Vector Store

We use `text-embedding-3-small` — cheap, fast, and accurate enough for this use case. `Chroma.from_documents()` handles batching internally. The store is persisted to disk so we don't re-embed on every run.

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

print("Building vector store — this may take a moment...")

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
    persist_directory=str(VECTORSTORE_DIR),
    collection_name="youtube_qa",
)

print(f"Vector store built and persisted to {VECTORSTORE_DIR.resolve()}")
print(f"Total documents in store: {vectorstore._collection.count()}")

Building vector store — this may take a moment...
Vector store built and persisted to /home/lopoa/projects/Project-Youtube-QA-Bot/data/vectorstore
Total documents in store: 514


## 5. Load the Persisted Store (for future runs)

After the initial build, always load from disk instead of re-embedding. Comment out the build cell above once the store exists.

In [6]:
# Uncomment this block after the store is built — use instead of the build cell above

# from langchain_openai import OpenAIEmbeddings
# from langchain_chroma import Chroma

# embedding_model = OpenAIEmbeddings(
#     model="text-embedding-3-small",
#     openai_api_key=os.getenv("OPENAI_API_KEY"),
# )

# vectorstore = Chroma(
#     persist_directory=str(VECTORSTORE_DIR),
#     embedding_function=embedding_model,
#     collection_name="youtube_qa",
# )

# print(f"Loaded store with {vectorstore._collection.count()} documents")

## 6. Build the Retriever

`k=4` returns the four most relevant chunks per query. This is a good default — enough context for GPT-4o-mini without blowing up the prompt.

In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print("Retriever ready.")
print("Search type: similarity | k: 4")

Retriever ready.
Search type: similarity | k: 4


## 7. Smoke Test the Retriever

Run a few test queries across different categories and inspect what comes back. We want to confirm the right chunks surface and that metadata is intact.

In [8]:
test_queries = [
    "What is machine learning?",
    "How do neural networks work?",
    "What did Erwin say in his speech?",
]

for query in test_queries:
    print(f"Query: {query}")
    results = retriever.invoke(query)
    for i, doc in enumerate(results):
        print(f"  [{i+1}] [{doc.metadata['category']}] {doc.metadata['title']}")
        print(f"  chunk_id={doc.metadata['chunk_id']} | {len(doc.page_content)} chars")
        print(f"  {doc.page_content[:120].strip()}...")
    print()

Query: What is machine learning?
  [1] [education] neural_networks_explained
  chunk_id=34 | 284 chars
  . By the way, so much of machine learning just comes down to having a good grasp of linear algebra, so for any of you wh...
  [2] [education] transformers_explained
  chunk_id=33 | 271 chars
  . So a common challenge that those of you working in machine learning will be familiar with is just getting the labeled...
  [3] [tech_ai] rag_explained
  chunk_id=16 | 497 chars
  so really you shouldn't think about it as pattern matching and just trying to predict the next word yes it was trained j...
  [4] [education] neural_networks_explained
  chunk_id=29 | 375 chars
  . All said and done, this network has almost exactly 13,000 total weights and biases. 13,000 knobs and dials that can be...

Query: How do neural networks work?
  [1] [education] neural_networks_explained
  chunk_id=11 | 437 chars
  . And of course the heart of the network as an information processing mechanism comes down t

## 8. Category-Filtered Retrieval (Optional)

ChromaDB supports metadata filtering out of the box. Useful later when you want to scope a query to a specific category — for example, only searching tech/AI videos.

In [9]:
filtered_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {"category": "tech_ai"},
    },
)

query = "What are the latest AI breakthroughs?"
results = filtered_retriever.invoke(query)

print(f"Filtered query: {query}")
for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.metadata['title']} | chunk {doc.metadata['chunk_id']}")
    print(f"{doc.page_content[:120].strip()}...")

Filtered query: What are the latest AI breakthroughs?
[1] rag_explained | chunk 5
in the in this last few months is that we are seeing you know the premise of something that looks like artificial genera...
[2] openai_gpt4_announcement | chunk 0
[Applause] hi listeners welcome back to no priors today we're hanging out with Andre karpathy who needs no introduction...
[3] rag_explained | chunk 113
it data and it will do analysis for you it can be used as a privacy detector it's medical and law knowledge is amazing a...
[4] openai_gpt4_announcement | chunk 64
brain um and it's just not there yet how do you think about um human augmentation with different AI systems over time do...
